### Installation

```bash
pip install yfinance gnews torch vaderSentiment
```

### What Each Package Does

| Package | Role | Needs Key? |
|---|---|---|
| `yfinance` | Download historical OHLCV stock data from Yahoo Finance | No |
| `gnews` | Scrape Google News RSS for headlines by keyword and date | No |
| `vaderSentiment` | Rule-based sentiment scorer, works offline, no GPU needed | No |
| `torch` | Build and train RNN / LSTM / GRU models | No |
| `scikit-learn` | MinMaxScaler and evaluation metrics | No |

In [ ]:
pip install yfinance gnews torch vaderSentiment

## Overview & Motivation

Stock prices are shaped by two very different kinds of information:

- **Quantitative signals** — price, volume, returns, moving averages. These are clean numbers that are easy to feed into a model.
- **Qualitative signals** — news articles, earnings announcements, analyst opinions. These are messy text that first need to be converted into numbers (sentiment scores) before a model can use them.

A **Multivariate Time Series (MVTS)** model eats both kinds at once and learns how they interact over time.

### What We Are Building

```
Google News Headlines ──► GNews (free, no key) ──► Sentiment Score ──┐
                                                                        ├──► MVTS Model ──► Next-Day Close
yfinance OHLCV Data ──────────────────────────► Engineered Features ──┘
```

The full pipeline has three stages:

```
Stage 1: Data Collection      Stage 2: Feature Engineering     Stage 3: Modeling
──────────────────────        ──────────────────────────────    ──────────────────
yfinance  ──► stock_df        Returns, MAs, Volume change  ──►  SimpleRNN
GNews     ──► news_df         VADER sentiment score        ──►  SimpleLSTM
                              Merge on Date index          ──►  SimpleGRU
                                                                    │
                                                               Evaluate & Compare
```

### Why RNN, LSTM, and GRU?

Recurrent models process sequences one time step at a time, passing a hidden state forward. This makes them natural fits for time series where yesterday's data influences today's prediction.

| Model | Core Idea | Strength | Weakness |
|---|---|---|---|
| **RNN** | Recurrent hidden state updated at every step | Simple, fast, good baseline | Vanishing gradient on long sequences |
| **LSTM** | Cell state + 3 gates (forget, input, output) | Captures long-range dependencies | More parameters, slower to train |
| **GRU** | Merged state + 2 gates (reset, update) | Fast, fewer params than LSTM | Slightly less expressive than LSTM |

All three have identical PyTorch interfaces — swapping one for another is a single-line change.

---

In [ ]:
# ── CONFIG — change these to run on any ticker ────────────────────────────
TICKER     = "AAPL"          # Stock ticker symbol
COMPANY    = "Apple"         # Used as the GNews search keyword
START_DATE = "2024-01-01"    # Start of the data window
END_DATE   = "2026-04-03"    # End of the data window
SEQ_LEN    = 20              # Look-back window: 20 trading days ~ 1 month
BATCH_SIZE = 32
EPOCHS     = 50
LR         = 1e-3
HIDDEN     = 64
LAYERS     = 2
DROPOUT    = 0.2

## Step 1 — Download Daily Stock Data

We use `yfinance` to pull OHLCV (Open, High, Low, Close, Volume) data and add a few simple technical features on top.

```python
import yfinance as yf
import pandas as pd
import numpy as np

stock_df = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=True)
stock_df = stock_df[["Open", "High", "Low", "Close", "Volume"]].copy()
stock_df.index = pd.to_datetime(stock_df.index)
stock_df.index.name = "Date"

stock_df["Return"]     = stock_df["Close"].pct_change()
stock_df["Range"]      = stock_df["High"] - stock_df["Low"]
stock_df["MA_5"]       = stock_df["Close"].rolling(5).mean()
stock_df["MA_20"]      = stock_df["Close"].rolling(20).mean()
stock_df["Vol_Change"] = stock_df["Volume"].pct_change()

stock_df.dropna(inplace=True)
```

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

stock_df = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=True)
stock_df = stock_df[["Open", "High", "Low", "Close", "Volume"]].copy()
stock_df.index = pd.to_datetime(stock_df.index)
stock_df.index.name = "Date"

stock_df["Return"]     = stock_df["Close"].pct_change()
stock_df["Range"]      = stock_df["High"] - stock_df["Low"]
stock_df["MA_5"]       = stock_df["Close"].rolling(5).mean()
stock_df["MA_20"]      = stock_df["Close"].rolling(20).mean()
stock_df["Vol_Change"] = stock_df["Volume"].pct_change()

stock_df.dropna(inplace=True)

## Step 2 — Download News Headlines with GNews

### GNews

`gnews` is a lightweight Python package that reads the **Google News RSS feed** — no account, no API key, no rate-limit fees. Google News aggregates articles from thousands of publishers worldwide, making it a rich source for financial headlines.

```
GNews  ──►  Google News RSS  ──►  JSON list of articles
              (free, open)          {title, description, published date, url}
```

In [ ]:
import time
from gnews import GNews

date_range = pd.date_range(start=START_DATE, end=END_DATE, freq="B")
records = []

for i, date in enumerate(date_range):
    next_day = date + pd.Timedelta(days=1)
    gn = GNews(
        language    = "en",
        country     = "US",
        max_results = 10,
        start_date  = (date.year, date.month, date.day),
        end_date    = (next_day.year, next_day.month, next_day.day),
    )
    articles = gn.get_news(f"{COMPANY} {TICKER} stock")
    records.append({
        "Date"      : pd.Timestamp(date.strftime("%Y-%m-%d")),
        "Headlines" : [a["title"] for a in articles if a.get("title")],
    })
    print(f"  {i + 1}/{len(date_range)} — {date.date()}")
    time.sleep(2)

news_df = pd.DataFrame(records).set_index("Date")

In [ ]:
# Check a sample day
sample_date = "2026-03-02"
print(f"Headlines on {sample_date}:")
for h in news_df.loc[sample_date, "Headlines"]:
    print(f"  - {h}")

# Count headlines per day
news_df["n_headlines"] = news_df["Headlines"].apply(len)
print(f"\nAverage headlines per day : {news_df['n_headlines'].mean():.1f}")
print(f"Days with zero headlines  : {(news_df['n_headlines'] == 0).sum()}")
print(f"Max headlines in a day    : {news_df['n_headlines'].max()}")

In [ ]:
import json

# ── Save ───────────────────────────────────────────────────────────────────
#news_df["Headlines_json"] = news_df["Headlines"].apply(json.dumps)
#news_df[["Headlines_json"]].to_csv("raw_news.csv")
#print("Saved to raw_news.csv")

# ── Reload ─────────────────────────────────────────────────────────────────
def load_news_csv(path="raw_news.csv"):
    df = pd.read_csv(path, index_col="Date", parse_dates=True)
    df["Headlines"] = df["Headlines_json"].apply(json.loads)
    return df[["Headlines"]]

news_df = load_news_csv("raw_news.csv")   # uncomment after first run

## Step 3 — Sentiment Analysis on Headlines

We convert raw text headlines into a single numeric score per day using **VADER** — a rule-based sentiment analyzer that requires no GPU and no model download.

### How VADER Works

VADER assigns each word a valence score from its built-in lexicon. The **compound score** summarises the overall sentiment of a sentence on a scale from -1.0 to +1.0.

```
"Apple beats earnings expectations"   -->  compound =  0.72  (positive)
"iPhone sales disappoint investors"   -->  compound = -0.61  (negative)
"Apple holds annual developer event"  -->  compound =  0.00  (neutral)
```

For each day we average the compound scores of all its headlines.

```python
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()

def compute_daily_sentiment(headlines):
    """
    Average VADER compound score for a list of headlines.
    Returns 0.0 (neutral) when there are no headlines.

    Compound score range:  -1.0 (very negative)  to  +1.0 (very positive)
    """
    if not headlines:
        return 0.0   # no news = treat as neutral

    scores = [vader.polarity_scores(h)["compound"] for h in headlines]
    return sum(scores) / len(scores)


news_df["Sentiment"] = news_df["Headlines"].apply(compute_daily_sentiment)

print("Sentiment statistics:")
print(news_df["Sentiment"].describe().round(4))
```

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()

def compute_daily_sentiment(headlines):
    if not headlines:          # handles empty list / None
        return 0.0
    scores = [vader.polarity_scores(h)["compound"] for h in headlines if h]
    return sum(scores) / len(scores) if scores else 0.0

news_df["Sentiment"] = news_df["Headlines"].apply(compute_daily_sentiment)

print(news_df["Sentiment"].describe().round(4))

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.hist(news_df["Sentiment"], bins=40, color="steelblue", edgecolor="white", alpha=0.85)
ax1.set(title="Distribution of Daily Sentiment Scores", xlabel="VADER Compound Score", ylabel="Count")
ax1.legend()

ax2.fill_between(news_df.index, news_df["Sentiment"], where=news_df["Sentiment"] >= 0, color="seagreen", alpha=0.6, label="Positive")
ax2.fill_between(news_df.index, news_df["Sentiment"], where=news_df["Sentiment"] <  0, color="tomato",   alpha=0.6, label="Negative")
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set(title="Daily Sentiment Over Time", xlabel="Date", ylabel="Compound Score")
ax2.legend()

plt.tight_layout()
plt.savefig("sentiment_overview.png", dpi=120)
plt.show()

In [ ]:
# Flatten MultiIndex and build dataset
stock_df.columns = stock_df.columns.get_level_values(0)
stock_df.index = pd.to_datetime(stock_df.index).normalize()
news_df.index  = pd.to_datetime(news_df.index).normalize()

dataset = stock_df.join(news_df[["Sentiment"]], how="left")
dataset["Sentiment"] = dataset["Sentiment"].fillna(0.0)
dataset["Target"]    = dataset["Close"].shift(-1)
dataset.dropna(inplace=True)

print(f"Shape   : {dataset.shape}")
print(f"Columns : {dataset.columns.tolist()}")
print(dataset.head(3))

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(15, 14), sharex=True)

ax1.plot(dataset.index, dataset["Close"], color="steelblue", linewidth=1.5)
ax1.plot(dataset.index, dataset["MA_5"],  color="orange", linewidth=1, linestyle="--", label="MA 5")
ax1.plot(dataset.index, dataset["MA_20"], color="red",    linewidth=1, linestyle="--", label="MA 20")
ax1.set(title=f"{TICKER} — Close Price with Moving Averages", ylabel="Price ($)")
ax1.legend(loc="upper left")

colors = ["seagreen" if r >= 0 else "tomato" for r in dataset["Return"]]
ax2.bar(dataset.index, dataset["Return"], color=colors, alpha=0.7, width=1)
ax2.axhline(0, color="black", linewidth=0.7)
ax2.set(title="Daily Return", ylabel="Return")

ax3.bar(dataset.index, dataset["Volume"] / 1e6, color="mediumpurple", alpha=0.6, width=1)
ax3.set(title="Daily Volume", ylabel="Volume (M)")

ax4.fill_between(dataset.index, dataset["Sentiment"], where=dataset["Sentiment"] >= 0, color="seagreen", alpha=0.55, label="Positive")
ax4.fill_between(dataset.index, dataset["Sentiment"], where=dataset["Sentiment"] <  0, color="tomato",   alpha=0.55, label="Negative")
ax4.axhline(0, color="black", linewidth=0.7)
ax4.set(title="News Sentiment", ylabel="Compound Score", xlabel="Date")
ax4.legend()

plt.tight_layout()
plt.savefig("eda_overview.png", dpi=120)
plt.show()

In [ ]:
FEATURE_COLS = [
    "Open", "High", "Low", "Close", "Volume",
    "Return", "Range", "MA_5", "MA_20", "Vol_Change",
    "Sentiment",       # the GNews-derived feature
]
TARGET_COL = "Target"
N_FEATURES = len(FEATURE_COLS)   # 11

print(f"Number of features : {N_FEATURES}")
print(f"Feature list       : {FEATURE_COLS}")

In [ ]:
n       = len(dataset)
n_train = int(n * 0.80)
n_val   = int(n * 0.10)
# the remaining 10% is the test set

train_df = dataset.iloc[:n_train]
val_df   = dataset.iloc[n_train : n_train + n_val]
test_df  = dataset.iloc[n_train + n_val :]

print(f"Train : {len(train_df)} rows  ({train_df.index[0].date()} to {train_df.index[-1].date()})")
print(f"Val   : {len(val_df)}  rows  ({val_df.index[0].date()} to {val_df.index[-1].date()})")
print(f"Test  : {len(test_df)} rows  ({test_df.index[0].date()} to {test_df.index[-1].date()})")

In [ ]:
from sklearn.preprocessing import MinMaxScaler

feature_scaler = MinMaxScaler()
target_scaler  = MinMaxScaler()

# Fit on train, transform all splits
X_train = feature_scaler.fit_transform(train_df[FEATURE_COLS])
X_val   = feature_scaler.transform(val_df[FEATURE_COLS])
X_test  = feature_scaler.transform(test_df[FEATURE_COLS])

y_train = target_scaler.fit_transform(train_df[[TARGET_COL]])
y_val   = target_scaler.transform(val_df[[TARGET_COL]])
y_test  = target_scaler.transform(test_df[[TARGET_COL]])

print(f"X_train shape: {X_train.shape}  -- (rows, features)")
print(f"y_train shape: {y_train.shape}  -- (rows, 1)")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class SequenceDataset(Dataset):
    def __init__(self, X, y, seq_len):
        self.X, self.y, self.seq_len = torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32), seq_len

    def __len__(self):
        return len(self.X) - self.seq_len

    def __getitem__(self, idx):
        return self.X[idx : idx + self.seq_len], self.y[idx + self.seq_len]


train_ds, val_ds, test_ds = SequenceDataset(X_train, y_train, SEQ_LEN), SequenceDataset(X_val, y_val, SEQ_LEN), SequenceDataset(X_test, y_test, SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

In [ ]:
import torch.nn as nn

class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True,
                          dropout=dropout, nonlinearity="tanh")
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

In [ ]:
class SimpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

In [ ]:
class SimpleGRU(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

In [ ]:
import torch.optim as optim

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR):
    model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    history   = {"train_loss": [], "val_loss": []}

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            batch_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                val_losses.append(criterion(model(X_batch), y_batch).item())

        avg_train, avg_val = sum(batch_losses) / len(batch_losses), sum(val_losses) / len(val_losses)
        history["train_loss"].append(avg_train)
        history["val_loss"].append(avg_val)

        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {avg_train:.6f} | Val Loss: {avg_val:.6f}")

    return model, history

In [ ]:
rnn_model  = SimpleRNN(input_size=N_FEATURES,  hidden_size=HIDDEN, num_layers=LAYERS, dropout=DROPOUT)
lstm_model = SimpleLSTM(input_size=N_FEATURES, hidden_size=HIDDEN, num_layers=LAYERS, dropout=DROPOUT)
gru_model  = SimpleGRU(input_size=N_FEATURES,  hidden_size=HIDDEN, num_layers=LAYERS, dropout=DROPOUT)

print("=" * 55)
print(" Training Vanilla RNN")
print("=" * 55)
rnn_model, rnn_history = train_model(rnn_model, train_loader, val_loader)

print("\n" + "=" * 55)
print(" Training LSTM")
print("=" * 55)
lstm_model, lstm_history = train_model(lstm_model, train_loader, val_loader)

print("\n" + "=" * 55)
print(" Training GRU")
print("=" * 55)
gru_model, gru_history = train_model(gru_model, train_loader, val_loader)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate_model(model, test_loader, target_scaler):
    model.eval()
    with torch.no_grad():
        preds  = np.vstack([model(X.to(DEVICE)).cpu().numpy() for X, _ in test_loader])
        actual = np.vstack([y.numpy() for _, y in test_loader])

    preds  = target_scaler.inverse_transform(preds)
    actual = target_scaler.inverse_transform(actual)

    return {
        "MAE"      : mean_absolute_error(actual, preds),
        "RMSE"     : mean_squared_error(actual, preds) ** 0.5,
        "MAPE (%)" : np.mean(np.abs((actual - preds) / actual)) * 100,
        "preds"    : preds,
        "actual"   : actual,
    }

rnn_results  = evaluate_model(rnn_model,  test_loader, target_scaler)
lstm_results = evaluate_model(lstm_model, test_loader, target_scaler)
gru_results  = evaluate_model(gru_model,  test_loader, target_scaler)

In [ ]:
results_df = pd.DataFrame({
    "Model"   : ["RNN", "LSTM", "GRU"],
    "MAE ($)" : [rnn_results["MAE"],      lstm_results["MAE"],      gru_results["MAE"]],
    "RMSE($)" : [rnn_results["RMSE"],     lstm_results["RMSE"],     gru_results["RMSE"]],
    "MAPE(%)" : [rnn_results["MAPE (%)"], lstm_results["MAPE (%)"], gru_results["MAPE (%)"]],
}).set_index("Model").round(4)

print(results_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, (history, name, color) in zip(axes, [
    (rnn_history,  "RNN",  "tomato"),
    (lstm_history, "LSTM", "steelblue"),
    (gru_history,  "GRU",  "seagreen"),
]):
    epochs = range(1, len(history["train_loss"]) + 1)
    ax.plot(epochs, history["train_loss"], color=color, linewidth=2, label="Train")
    ax.plot(epochs, history["val_loss"],   color=color, linewidth=2, label="Validation", linestyle="--", alpha=0.75)
    ax.set(title=f"{name} Loss Curves", xlabel="Epoch", ylabel="MSE Loss")
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("Training vs Validation Loss", y=1.02)
plt.tight_layout()
plt.savefig("loss_curves.png", dpi=120)
plt.show()

In [ ]:
test_dates = test_df.index[SEQ_LEN:]
fig, axes  = plt.subplots(3, 1, figsize=(16, 13), sharex=True)

for ax, (results, name, color) in zip(axes, [
    (rnn_results,  "RNN",  "tomato"),
    (lstm_results, "LSTM", "steelblue"),
    (gru_results,  "GRU",  "seagreen"),
]):
    actual, preds = results["actual"].flatten(), results["preds"].flatten()
    ax.plot(test_dates, actual, color="black", linewidth=1.5, label="Actual")
    ax.plot(test_dates, preds,  color=color,   linewidth=1.5, label=name, alpha=0.85)
    ax.fill_between(test_dates, actual, preds, alpha=0.1, color=color)
    ax.set(title=f"{name} | MAE: ${results['MAE']:.2f}  RMSE: ${results['RMSE']:.2f}  MAPE: {results['MAPE (%)']:.2f}%", ylabel="Price ($)")
    ax.legend(loc="upper left")
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("Date")
plt.suptitle(f"{TICKER} Forecast Comparison", y=1.01)
plt.tight_layout()
plt.savefig("predictions.png", dpi=120)
plt.show()

#### Univariate Model

```python
# ── Univariate setup ──────────────────────────────────────────────
N_FEATURES = 1

scaler  = MinMaxScaler()
X_train = scaler.fit_transform(train_df[["Close"]])
X_val   = scaler.transform(val_df[["Close"]])
X_test  = scaler.transform(test_df[["Close"]])

y_train = X_train.copy()
y_val   = X_val.copy()
y_test  = X_test.copy()
```

##### Run everything from Data Scaling

### Explainability using SHAP
##### pip install shap

In [ ]:
import shap

background  = torch.tensor(X_train[:100], dtype=torch.float32).unsqueeze(1).expand(-1, SEQ_LEN, -1).to(DEVICE)
test_input  = torch.tensor(X_test[:50],  dtype=torch.float32).unsqueeze(1).expand(-1, SEQ_LEN, -1).to(DEVICE)

explainer   = shap.GradientExplainer(lstm_model, background)
shap_values = explainer.shap_values(test_input)  # (samples, seq_len, features)

mean_shap = np.abs(shap_values).mean(axis=(0, 1)).squeeze()
shap_df   = pd.Series(mean_shap, index=FEATURE_COLS).sort_values(ascending=False)
print(shap_df.round(4))